## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Ministry of Transport validation, stage 3: counts and transit (T12, T14, T15, T17)

**Question.** How do the 2022 layers compare, in the guideline's format, with traffic counts (cordons), with RavKav boardings by
origin zone, with rail station demand and with measured bus running times?

**Not run here (see `docs/MOT_VALIDATION_PLAN.md`).** The link-by-link comparison with the 3,241 counts (T11) needs the all-or-nothing
assignment of step 42; the road travel-time check (T13) needs the Google sample (step 37); Metronit link loads need alightings the 2025 files lack.

**Honest framing.** The car layer holds *residents'* person trips by car with both ends in the study area, in the AM period only, with no
trucks, taxis, buses or trips with an end outside the study area. The guideline's ±15 % cordon criterion presumes a complete vehicle
demand. A miss is therefore expected where the counts are mostly measured, and is reported as a miss; no parameter is changed.

In [2]:
import json, shapely
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
CRS = 2039
PCE = 1.10
OCC_STEP36 = 1.33               # the AM value of step 36 (and of the first version of this test): driver + passenger trips over driver trips with the hour taken from the row of the ARRIVAL place
# Corrected 4 October 2026 (wave 2, T10): the departure hour of a trip is the Dep_h of the PREVIOUS place of the person-day (arrtime - previous deptime = TrvlTime), as in steps 15 and 16.
# The step-36 value used the Dep_h of the row itself, which is the hour the person leaves the destination. The window's own occupancy is therefore recomputed here for all three windows.
_tr = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx').sort_values(['PerID3', 'SurveyDay', 'placeno']); _g = _tr.groupby(['PerID3', 'SurveyDay'])
_tr['trip_dep_h'] = _g['Dep_h'].shift(1); _tr['_o'] = _g['actTaz'].shift(1); _w = _tr.dropna(subset=['trip_dep_h', '_o']); _w = _w[_w.trip_dep_h.isin(HOURS)]
OCC = float(_w[_w.mainmode.isin([10, 11])]['new_wf'].sum() / _w[_w.mainmode == 10]['new_wf'].sum())
if PERIOD == 'AM':
    PHF_CAR_STUDY = float(pd.read_csv('Output/ths2017/three_mode_2022/peak_hour_factors.csv').query("level == 'study area, all trips' and layer == 'CAR'")['PHF3h'].iloc[0])
else:
    PHF_CAR_STUDY = np.nan      # no peak-hour factor is applied outside the AM
print(f'{PLABEL}: occupancy {OCC:.3f} (step-36 value for the AM was {OCC_STEP36}), PCE factor {PCE}, survey car peak-hour factor (study area, AM only) {PHF_CAR_STUDY:.3f}')

PM 16:00-19:00: occupancy 1.518 (step-36 value for the AM was 1.33), PCE factor 1.1, survey car peak-hour factor (study area, AM only) nan


## 1. T12 - cordons and screenlines in the guideline's format (guideline 5.1)

**Test.** Survey car vehicles against counted vehicles on closed cordons, split into directional sectors (2-4 per cordon, by the bearing of the
outside end from the cordon centroid: N, E, S, W), inbound and outbound, for 06:00-09:00 and the 07:00-08:00 peak hour. **Criterion:** within ±15 %
of the count (±10 % for an external belt, none here), and RMSE% below 35 % across the cells. The count side is the table of step 36
(crossing car links; links without a count take the median PCE per lane of the counted links of the same road type; PCE 1.10); the demand side
is the 2022 car layer ÷ the window occupancy (section 0) plus the through trips (both ends outside, straight desire line crossing the cordon, counted once at the entry
point and once at the exit point). Peak hour: the count side is the 07:00-08:00 hour of the same links; the survey side is the three hours times the
study-area car factor from step 20.

In [3]:
taz_shp = gpd.read_file('Input/TAZ_North/TAZ_North.shp').to_crs(CRS)
taz_shp['TAZ'] = taz_shp['TAZ_NUMBER'].astype(int); taz_shp['SZ'] = taz_shp['SUPERZONE'].astype(int)
car = pd.read_csv(f'Output/ths2017/three_mode_2022{SUF}/car_2022_taz.csv', index_col=0); car.index = car.index.astype(int); car.columns = car.columns.astype(int)
TAZ = list(car.index)
rp = taz_shp.drop_duplicates('TAZ').set_index('TAZ').geometry.representative_point()
cx = rp.reindex(TAZ).x.values; cy = rp.reindex(TAZ).y.values
links = pd.read_csv('Output/validation/car_cordon_crossing_links.csv')
CORDONS = {'Haifa city': [8, 12, 13, 14, 15, 16, 17], 'Krayot': [5, 6, 7], 'Kiryat Ata + Zevulun': [9], 'Nazareth + Nof HaGalil': [19, 20],
           'Tirat Carmel': [24], 'Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)': [5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18]}
SECT = ['north', 'east', 'south', 'west']
def sector_of(dx, dy):
    b = (np.degrees(np.arctan2(dx, dy)) + 360) % 360               # bearing clockwise from north
    return SECT[int(((b + 45) % 360) // 90)]

nz = np.argwhere(car.values > 0); o_idx, d_idx = nz[:, 0], nz[:, 1]; vals = car.values[o_idx, d_idx]
desire = shapely.linestrings(np.stack([np.stack([cx[o_idx], cy[o_idx]], axis=1), np.stack([cx[d_idx], cy[d_idx]], axis=1)], axis=1))
CV, SV, DV, RV, WV = f'count vehicles {W}', f'survey vehicles {W}', f'diff % {W}', f'ratio {W}', f'within ±15 % ({W})'
rows = []
for name, szs in CORDONS.items():
    poly = taz_shp[taz_shp.SZ.isin(szs)].geometry.union_all(); cen = poly.centroid
    inside = set(taz_shp.loc[taz_shp.SZ.isin(szs), 'TAZ']) & set(TAZ)
    in_mask = np.array([z in inside for z in TAZ])
    sec_taz = np.array([sector_of(cx[i] - cen.x, cy[i] - cen.y) for i in range(len(TAZ))])
    P = {(s, d): 0.0 for s in SECT for d in ['inbound', 'outbound']}; T = {k: 0.0 for k in P}
    for i in range(len(TAZ)):
        pass
    # one-end trips by the sector of the outside zone
    oi, di = o_idx, d_idx
    for direction, sel, out_ix in [('inbound', (~in_mask[oi]) & in_mask[di], oi), ('outbound', in_mask[oi] & (~in_mask[di]), di)]:
        for s in SECT:
            m = sel & (sec_taz[out_ix] == s); P[(s, direction)] = float(vals[m].sum())
    # through trips: entry and exit points of the desire line on the cordon boundary
    both_out = (~in_mask[oi]) & (~in_mask[di]); cand = np.where(both_out)[0]
    hit = cand[shapely.intersects(desire[cand], poly)]
    bnd = poly.boundary
    for k in hit:
        pts = desire[k].intersection(bnd)
        pts = [p for p in (pts.geoms if hasattr(pts, 'geoms') else [pts]) if p.geom_type == 'Point']
        if not pts: continue
        po, pd_ = Point(cx[oi[k]], cy[oi[k]]), Point(cx[di[k]], cy[di[k]])
        ent = min(pts, key=lambda p: p.distance(po)); ext = min(pts, key=lambda p: p.distance(pd_))
        T[(sector_of(ent.x - cen.x, ent.y - cen.y), 'inbound')] += vals[k]; T[(sector_of(ext.x - cen.x, ext.y - cen.y), 'outbound')] += vals[k]
    # count side: crossing links, sector of the outside end point
    L = links[links.cordon == name].copy()
    def out_pt(r):
        g = wkt.loads(r['geometry']); c = g.coords
        return Point(c[0]) if r['direction'] == 'inbound' else Point(c[-1])
    L['sector'] = [sector_of(out_pt(r).x - cen.x, out_pt(r).y - cen.y) for _, r in L.iterrows()]
    if PERIOD == 'AM': L['pce_period'] = L['pce_imputed']                  # counted links: their count; others: median PCE per lane of the cordon (step 36)
    else:                                                                  # same rule for the window's hours: uncounted links scaled by the cordon's counted window / AM ratio
        hcs = [f'YARAM{h}' for h in HOURS]; cn = L[L.counted]; ratio = cn[hcs].sum().sum() / cn['pce_6_9'].sum()
        L['pce_period'] = np.where(L.counted, L[hcs].sum(axis=1), L['pce_imputed'] * ratio)
    for s in SECT:
        for direction in ['inbound', 'outbound']:
            x = L[(L.sector == s) & (L.direction == direction)]
            surv_p = P[(s, direction)] + T[(s, direction)]
            if len(x) == 0 and surv_p == 0: continue
            pce_i, pce_c = x['pce_period'].sum(), x.loc[x.counted, 'pce_period'].sum()
            rows.append({'cordon': name, 'sector': s, 'direction': direction, 'crossing links': len(x), 'counted links': int(x.counted.sum()),
                         'counted share of PCE': pce_c / pce_i if pce_i else np.nan,
                         CV: pce_i / PCE, 'count vehicles 07-08': (x['YARAM7_imp'].sum() / PCE if PERIOD == 'AM' else np.nan),
                         'survey person trips one-end': P[(s, direction)], 'survey person trips through': T[(s, direction)],
                         SV: surv_p / OCC, 'survey vehicles 07-08': surv_p / OCC * PHF_CAR_STUDY})
t12 = pd.DataFrame(rows)
t12[DV] = vm.pct_diff(t12[CV], t12[SV]); t12['diff % 07-08'] = vm.pct_diff(t12['count vehicles 07-08'], t12['survey vehicles 07-08'])
t12[RV] = t12[SV] / t12[CV]
t12[WV] = t12[DV].abs() <= vm.CRIT['screenline_pct']['other']
t12['within ±15 % (07-08)'] = t12['diff % 07-08'].abs() <= vm.CRIT['screenline_pct']['other']
t12.to_csv(vfile('T12_cordon_sectors'), index=False, float_format='%.3f')
# the cordon totals must reproduce step 36
tot = t12.groupby(['cordon', 'direction'])[[CV, SV]].sum(); tot['ratio'] = tot[SV] / tot[CV]
ref36 = pd.read_csv('Output/validation/car_cordon_counts.csv').set_index(['cordon', 'direction'])['ratio survey ÷ count']
tot['step 36 ratio'] = ref36.reindex(tot.index).values
print('cordon totals from the sector table against step 36 (should agree):'); print(tot.round(3).to_string())
t12.round(2)

cordon totals from the sector table against step 36 (should agree):
                                                                 count vehicles 16-19  survey vehicles 16-19  ratio  step 36 ratio
cordon                                                direction                                                                   
Haifa city                                            inbound               79731.636              36719.350  0.461          0.802
                                                      outbound              80690.545              55657.323  0.690          0.482
Kiryat Ata + Zevulun                                  inbound               50763.282              30331.207  0.598          0.658
                                                      outbound              48328.779              31034.683  0.642          0.898
Krayot                                                inbound               50988.413              33734.425  0.662          0.754
               

,cordon,sector,direction,crossing links,counted links,counted share of PCE,count vehicles 16-19,count vehicles 07-08,survey person trips one-end,survey person trips through,survey vehicles 16-19,survey vehicles 07-08,diff % 16-19,diff % 07-08,ratio 16-19,within ±15 % (16-19),within ±15 % (07-08)
0,Haifa city,north,inbound,1,0,0.00,1403.69,NaN,3177.13,139.78,2185.30,NaN,55.68,NaN,1.56,False,False
1,Haifa city,north,outbound,1,0,0.00,1403.69,NaN,6832.22,192.27,4627.99,NaN,229.70,NaN,3.30,False,False
2,Haifa city,east,inbound,22,8,0.28,40131.96,NaN,18960.98,13287.85,21246.71,NaN,-47.06,NaN,0.53,False,False
3,Haifa city,east,outbound,22,8,0.35,41203.60,NaN,36081.69,14164.37,33103.93,NaN,-19.66,NaN,0.80,False,False
4,Haifa city,south,inbound,10,0,0.00,24616.84,NaN,15857.59,1400.22,11370.07,NaN,-53.81,NaN,0.46,False,False
5,Haifa city,south,outbound,10,0,0.00,24616.84,NaN,24245.42,929.13,16585.91,NaN,-32.62,NaN,0.67,False,False
6,Haifa city,west,inbound,6,4,0.90,13579.15,NaN,2429.04,481.04,1917.27,NaN,-85.88,NaN,0.14,False,False
7,Haifa city,west,outbound,6,4,0.90,13466.42,NaN,2009.99,23.12,1339.49,NaN,-90.05,NaN,0.10,False,False
8,Krayot,north,inbound,2,1,0.45,7650.37,NaN,2030.47,4060.73,4013.10,NaN,-47.54,NaN,0.52,False,False
9,Krayot,north,outbound,1,0,0.00,4197.65,NaN,4553.83,6528.08,7301.16,NaN,73.93,NaN,1.74,False,False


In [4]:
def cell_stats(df, col_c, col_s, label):
    fs = vm.fit_stats(df[col_c], df[col_s]); within = float((df[col_s].sub(df[col_c]).abs() / df[col_c] <= 0.15).mean())
    return {'set': label, 'cells': fs['n'], 'count total': fs['obs_total'], 'survey total': fs['mod_total'], 'total diff %': fs['total_diff_pct'], 'R2': fs['R2'], 'slope (origin)': fs['slope_origin'],
            'RMSE%': fs['RMSE_pct'], 'share of cells within ±15 %': within}
# cells with no crossing link have no count to compare with (a sector of the survey geometry that the road network does not cross); they stay in the table, not in the statistics
t12['testable'] = t12['crossing links'] > 0
print('sector cells without a crossing link (not testable):', int((~t12.testable).sum()), f'carrying {t12.loc[~t12.testable, SV].sum():,.0f} survey vehicles')
T = t12[t12.testable]
good = T[T['counted share of PCE'] >= 0.5]
st_rows = [cell_stats(T, CV, SV, f'all testable sector cells, {W}'), cell_stats(good, CV, SV, f'cells with >= 50 % of the PCE counted, {W}')]
if PERIOD == 'AM': st_rows += [cell_stats(T, 'count vehicles 07-08', 'survey vehicles 07-08', 'all testable sector cells, 07-08 peak hour'), cell_stats(good, 'count vehicles 07-08', 'survey vehicles 07-08', 'cells with >= 50 % counted, 07-08 peak hour')]
st = pd.DataFrame(st_rows)
st.to_csv(vfile('T12_cordon_summary'), index=False, float_format='%.3f'); print(st.round(3).to_string(index=False))
for _, r in st.iterrows():
    ok = (r['share of cells within ±15 %'] >= 0.999) and (r['RMSE%'] < vm.CRIT['RMSE_pct_aggregate_max'])
    add_summary('T12', '5.1', f'cordon sectors vs counts: {r["set"]}', f'{int(r["cells"])} cordon-sector-direction cells', 'each cell ±15 %; RMSE% < 35 %',
                f'survey / count = {r["survey total"] / r["count total"]:.2f} in total; {r["share of cells within ±15 %"]:.0%} of cells within ±15 %; RMSE% {r["RMSE%"]:.0f}; slope {r["slope (origin)"]:.2f}; R2 {r["R2"]:.2f}',
                vm.verdict(ok, explained=True),
                'Residents-only AM layer against all traffic (trucks, vans, taxis, buses, non-residents, trips with an end outside the study area and through traffic on Roads 2, 4, 70, 79 are not in the layer); '
                'sectors with most links imputed say more about the imputation than the layer. Reported, not adjusted.', 'car layer 2022 (survey 2017/18); counts 2017-2023')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), facecolor='white')
for ax, (df, ttl) in zip(axes, [(T, 'all testable sector cells'), (good, 'cells with at least half of the PCE counted')]):
    scatter_origin(ax, df[CV], df[SV], f'Cordon sectors, {PLABEL}\n{ttl}', 'counted vehicles (observed)', 'survey car vehicles (model)', color=BLUE)
save_fig(fig, 'mot_T12_cordon_sectors.png')

sector cells without a crossing link (not testable): 2 carrying 4,337 survey vehicles
                                         set  cells  count total  survey total  total diff %    R2  slope (origin)  RMSE%  share of cells within ±15 %
            all testable sector cells, 16-19     44   603335.360    366483.102       -39.257 0.582           0.588 57.984                        0.159
cells with >= 50 % of the PCE counted, 16-19     18   234180.084    116174.556       -50.391 0.221           0.456 69.527                        0.222


## 2. T14 - bus boardings by origin zone (guideline 5.2)

**Test.** The 2022 bus layer's origin totals against RavKav's May 2022 journey origins (RavKav's own alightings, AM 06:00-09:00): per
super-zone (±15 %), with RMSE%, slope and R2 across zones; and per TAZ, with RMSE% by size class using the guideline's station-size
thresholds applied to zone boardings (the layer has no bus lines, so the per-line row of the guideline cannot be run). The 2022 layer takes RavKav's
volumes in the origin x segment cells where coverage is credible (the **anchored** cells: equal by construction) and keeps the survey volume in the **guarded**
cells, so the test is read separately for super-zones that are mostly anchored and mostly guarded. RavKav 2025 origins are a second reference
(boardings 1.08 x 2022 study-wide).

In [5]:
bus22 = pd.read_csv(f'Output/ths2017/three_mode_2022{SUF}/bus_2022_taz.csv', index_col=0); bus22.index = bus22.index.astype(int); bus22.columns = bus22.columns.astype(int)
def lm(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    return m.reindex(index=bus22.index, columns=bus22.columns, fill_value=0).fillna(0)
rk22 = lm('Output/bus/bus_od_taz_avg.csv' if PERIOD == 'AM' else f'Output/bus/periods/bus_od_taz_avg_{PERIOD}.csv')
REFS = ['RavKav 2022'] + (['RavKav 2025'] if PERIOD == 'AM' else [])      # the 2025 layer exists for the AM window only
rk25 = lm('Output/ravkav_2025/bus_od_taz_2025.csv') if PERIOD == 'AM' else None
keys_raw = pd.read_csv('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv', encoding='windows-1255') if not is_pointer('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv') else None
if keys_raw is not None:
    taz_sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW'].reindex(bus22.index).astype(int)
else:
    zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID'); taz_sz = zon.loc[bus22.index, 'SZ_NEW'].astype(int)
orig = pd.DataFrame({'RavKav 2022': rk22.sum(axis=1), **({'RavKav 2025': rk25.sum(axis=1)} if PERIOD == 'AM' else {}), 'bus 2022 layer': bus22.sum(axis=1)})
oz = orig.groupby(taz_sz.values).sum()
seg = pd.read_csv(f'Output/ths2017/two_mode{SUF}/bus_calibration_factors_segments.csv')
g = seg.groupby('origin SZ').apply(lambda d: pd.Series({'survey trips in guarded cells': d.loc[d.guarded.astype(bool), 'survey BUS trips'].sum(), 'survey trips': d['survey BUS trips'].sum()}), include_groups=False)
oz['guarded share of survey volume'] = (g['survey trips in guarded cells'] / g['survey trips']).reindex(oz.index).fillna(0)
# what the segmented rule itself delivers per origin super-zone: segments with >= 5 sampled survey trips take their own RavKav / survey ratio,
# thinner segments ("origin-wide ratio" or "no survey trips") inherit the origin-wide decision, so their volume is survey x that factor and not RavKav's
seg['rule volume'] = np.where(seg['guarded'].astype(bool), seg['survey BUS trips'], seg['survey BUS trips'] * seg['factor applied'])
seg['thin'] = seg['basis'].astype(str).str.startswith('origin-wide') | seg['basis'].astype(str).str.startswith('no survey')
rule = seg.groupby('origin SZ').apply(lambda d: pd.Series({'rule total': d['rule volume'].sum(), 'RavKav in thin segments': d.loc[d.thin, 'RavKav journeys'].sum(),
                                                           'rule volume in thin segments': d.loc[d.thin, 'rule volume'].sum()}), include_groups=False)
oz['volume the rule gives'] = rule['rule total'].reindex(oz.index); oz['gap from thin segments'] = (rule['rule volume in thin segments'] - rule['RavKav in thin segments']).reindex(oz.index)
for ref in REFS:
    oz[f'diff % vs {ref}'] = vm.pct_diff(oz[ref], oz['bus 2022 layer']); oz[f'within ±15 % ({ref})'] = oz[f'diff % vs {ref}'].abs() <= 15
oz['class'] = np.where(oz['guarded share of survey volume'] > 0.5, 'mostly guarded (survey volume kept)', 'mostly anchored (RavKav volume)')
oz.index.name = 'super-zone'; oz.to_csv(vfile('T14_bus_origins_by_superzone'), float_format='%.2f')
rows = []
for label, d in [('all super-zones', oz), ('mostly anchored super-zones', oz[oz['class'].str.contains('anchored')]), ('mostly guarded super-zones', oz[oz['class'].str.contains('guarded')])]:
    for ref in REFS:
        fs = vm.fit_stats(d[ref], d['bus 2022 layer'])
        rows.append({'set': label, 'reference': ref, 'zones': fs['n'], 'RavKav total': fs['obs_total'], 'layer total': fs['mod_total'], 'total diff %': fs['total_diff_pct'],
                     'R2': fs['R2'], 'slope (origin)': fs['slope_origin'], 'RMSE%': fs['RMSE_pct'], 'share within ±15 %': float(d[f'within ±15 % ({ref})'].mean())})
t14 = pd.DataFrame(rows); t14.to_csv(vfile('T14_bus_origins_summary'), index=False, float_format='%.3f'); print(t14.round(3).to_string(index=False))
tz = orig.copy(); tz['SZ'] = taz_sz.values
t14z = []
for ref in REFS:
    keep = tz[ref] > 0; fs = vm.fit_stats(tz.loc[keep, ref], tz.loc[keep, 'bus 2022 layer'])
    t14z.append({'level': 'TAZ (RavKav origins > 0)', 'reference': ref, **fs})
    for r in vm.rmse_by_class(tz.loc[keep, ref], tz.loc[keep, 'bus 2022 layer'], vm.STATION_RMSE_CLASSES):
        t14z.append({'level': 'TAZ size class ' + r['class'], 'reference': ref, 'n': r['n'], 'obs_total': r['obs_mean'] * r['n'], 'RMSE_pct': r['RMSE_pct'], 'limit_pct': r['limit_pct'], 'pass': r['pass']})
t14z = pd.DataFrame(t14z); t14z.to_csv(vfile('T14_bus_origins_TAZ'), index=False, float_format='%.3f'); print(t14z.round(2).to_string(index=False))

                        set   reference  zones  RavKav total  layer total  total diff %    R2  slope (origin)   RMSE%  share within ±15 %
            all super-zones RavKav 2022     36      78490.25    88817.360        13.157 0.772           1.085  50.379               0.556
mostly anchored super-zones RavKav 2022     30      71827.50    71109.337        -1.000 0.978           0.994  12.028               0.667
 mostly guarded super-zones RavKav 2022      6       6662.75    17708.023       165.776 0.979           2.432 235.249               0.000
                   level   reference   n  obs_total  mod_total  total_diff_pct   R2  slope_origin  R2_origin   MAE   RMSE  RMSE_pct  limit_pct  pass
TAZ (RavKav origins > 0) RavKav 2022 713   78490.25   88410.35           12.64 0.65          0.95       0.72 56.89 142.16    129.13        NaN   NaN
    TAZ size class 0-100 RavKav 2022 507   14536.75        NaN             NaN  NaN           NaN        NaN   NaN    NaN    200.54       50.0 False
 

In [6]:
bad_anch = oz[(oz['class'].str.contains('anchored')) & (~oz['within ±15 % (RavKav 2022)'])]
for _, r in t14.iterrows():
    if r['reference'] != 'RavKav 2022': continue
    ok = r['share within ±15 %'] >= 0.999
    anch = 'anchored' in r['set']; guard = 'guarded' in r['set']
    if anch:
        gz = ', '.join(f'{int(i)} ({d:+.0f} %; thin segments account for {t:+.0f} of the {r - l:+.0f} trips)' for i, d, t, r, l in zip(bad_anch.index, bad_anch['diff % vs RavKav 2022'], bad_anch['gap from thin segments'], bad_anch['RavKav 2022'], bad_anch['bus 2022 layer']))
        why = ('Anchored cells take RavKav\'s volume only in segments with at least 5 sampled survey trips; thinner segments inherit the origin-wide ratio, so the origin total is not RavKav\'s even when nothing is guarded. '
               'Where no volume is guarded the rule reproduces the layer to within a trip (column "volume the rule gives"; guarded cells are grown to 2022 on top). Zones outside +/-15 %: ' + gz + '.')
        status = vm.verdict(ok, explained=True)
    elif guard:
        why = 'In guarded cells the survey volume is kept because RavKav records under half of it (coverage: operators, cash fares, un-geocoded stops, with the data provider). This is the rule working as designed, and the test of it.'
        status = vm.verdict(ok, explained=True)
    else:
        why = 'Mixture of the anchored and guarded zones: see the two rows below.'; status = vm.verdict(ok, explained=True)
    add_summary('T14', '5.2', f'bus origin totals by super-zone vs RavKav 2022: {r["set"]}', f'{int(r["zones"])} super-zones', 'each zone within ±15 %',
                f'{r["share within ±15 %"]:.0%} of zones within ±15 %; layer / RavKav = {r["layer total"] / r["RavKav total"]:.2f}; R2 {r["R2"]:.2f}, slope {r["slope (origin)"]:.2f}, RMSE% {r["RMSE%"]:.0f}',
                status, why, 'bus 2022 layer vs RavKav May 2022 / 2025')
fig, axes = plt.subplots(1, len(REFS), figsize=(5.5 * len(REFS), 4.6), facecolor='white', squeeze=False); axes = axes[0]
for ax, ref in zip(axes, REFS):
    scatter_origin(ax, oz[ref], oz['bus 2022 layer'], f'Bus origins by super-zone: layer vs {ref}', f'{ref} journeys (observed)', 'bus 2022 layer (model)', color=AQUA)
save_fig(fig, 'mot_T14_bus_origins.png')

## Not repeated outside the AM window

* **T15 (rail stations)** needs the 2025 RavKav rail taps re-extracted for the window (a pass over the 668 MB extract, step 34's rail part with other hours); the 2019 station matrix for the window is built in step 16 (`rail_station_smartcard_2022_taz.csv` in `three_mode_2022_pm` / `_md`).
* **T17 (running times)** needs the timetable skims and the measured speeds of the window (steps 29 and 30 with other hours; the speed file has hourly columns).
* The 2025 RavKav layer and the on-board pattern exist for the AM window only, so T14 and T7 use RavKav May 2022 as the reference outside the AM.

## Summary of this notebook against the guideline

In [7]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage3{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T12,5.1,cordon sectors vs counts: all testable sector ...,44 cordon-sector-direction cells,each cell ±15 %; RMSE% < 35 %,survey / count = 0.61 in total; 16% of cells w...,miss (explained)
1,T12,5.1,cordon sectors vs counts: cells with >= 50 % o...,18 cordon-sector-direction cells,each cell ±15 %; RMSE% < 35 %,survey / count = 0.50 in total; 22% of cells w...,miss (explained)
2,T14,5.2,bus origin totals by super-zone vs RavKav 2022...,36 super-zones,each zone within ±15 %,56% of zones within ±15 %; layer / RavKav = 1....,miss (explained)
3,T14,5.2,bus origin totals by super-zone vs RavKav 2022...,30 super-zones,each zone within ±15 %,67% of zones within ±15 %; layer / RavKav = 0....,miss (explained)
4,T14,5.2,bus origin totals by super-zone vs RavKav 2022...,6 super-zones,each zone within ±15 %,0% of zones within ±15 %; layer / RavKav = 2.6...,miss (explained)
